# Track 03: from IC to a controlled trading experiment

**Install the two files from this package into your EXISTING sprint clone:**
- `research/track03_economic_experiment.ipynb` — this notebook.
- `strategies/janice_leadlag.py` — causal portfolio builder and audit entry point.

If a strategy file already exists, inspect/merge rather than overwrite. Use the
same Python environment as your earlier notebooks. No earlier research notebook
or companion research module is imported. Your merged Part 1 helper and the
repository's `src.features`, `src.backtest` and `src.stats` must be available.

**Question:** do the scores generate economically useful returns after execution,
funding and trading costs? IC alone cannot answer this.

| Portfolio | What changes | Purpose |
|---|---|---|
| reversal | score = -4h alt return | Simple benchmark |
| catchup | score = -(alt return - beta*leader return) | Incremental value of beta |
| catchup_strong | Same catchup, trade only when strong at scheduled decisions | Incremental value of flow conditioning |

Everything else is shared: 18 alts, long/short neutrality, active gross notional 1,
8-hour decision grid, weight construction, deadband, fees and execution model.
There is no new beta-window experiment, signal sign flip, or hyperparameter search.

This is an exploratory economic follow-through after the IC results. Record the
portfolio rules in your PR before enabling the new performance run. The supplied
`compute_weights` remains the declared **catchup_strong** strategy, even if a
benchmark later scores better. The notebook does not automatically choose a winner.

**Verification:** 11 synthetic tests passed. Every code cell completed on synthetic
data using the fetched upstream engine, generating all30 comparisons and6 paired
contrasts. The default plan-only path was also checked. Outputs are cleared; no
real-panel performance or full official audit was run here.


In [ ]:
import sys
from pathlib import Path
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents)
           if (p/'src'/'backtest.py').is_file() and (p/'src'/'contrib'/'janice.py').is_file()),None)
if ROOT is None:
    raise FileNotFoundError('Open inside your sprint clone with the merged Part 1 helper.')
if not (ROOT/'strategies'/'janice_leadlag.py').is_file():
    raise FileNotFoundError('Copy the supplied strategies/janice_leadlag.py before running this notebook.')
sys.path.insert(0,str(ROOT))
import matplotlib.pyplot as plt
from IPython.display import display
print('Repository:',ROOT)


## 1. Research functions
The strategy module contains signal-to-weight code only: no file access, future
labels or backtest imports. The notebook imports the official `run_backtest` for
performance. This keeps execution/funding conventions consistent with the harness.


In [ ]:
from dataclasses import asdict,replace
import hashlib
import inspect
import json
import platform

import numpy as np
import pandas as pd
from pandas.testing import assert_series_equal

from src.backtest import run_backtest,tiered_fee_bps
from strategies.janice_leadlag import (TradingRules,build_portfolios,compute_weights,
    prepare_signals,scheduled_book,threshold_step,neutral_unit_gross)

VARIANTS=('reversal','catchup','catchup_strong')


def synthetic_economic_panel(n=1400,seed=41):
    rng=np.random.default_rng(seed)
    times=pd.date_range('2026-01-01',periods=n,freq='h',tz='UTC')
    names=['BTCUSDT','ETHUSDT']+[f'ALT{i:02d}USDT' for i in range(18)]
    leader=rng.normal(0,.003,n)
    r=leader[:,None]*np.linspace(.8,1.8,20)+rng.normal(0,.002,(n,20))
    close=100*np.exp(np.cumsum(r,axis=0));op=np.vstack([close[:1],close[:-1]])
    quote=rng.uniform(100,10000,(n,20))
    d=pd.MultiIndex.from_product([times,names],names=['timestamp','symbol']).to_frame(index=False)
    d['open']=op.ravel();d['close']=close.ravel()
    d['high']=np.maximum(op,close).ravel()*1.001
    d['low']=np.minimum(op,close).ravel()*.999
    d['quote_volume']=quote.ravel();d['volume']=(quote/close).ravel()
    d['taker_buy_quote_volume']=(quote*rng.uniform(.03,.97,(n,20))).ravel()
    d['trades']=100
    funding=rng.normal(0,.00005,(n,20))
    funding[np.arange(n)%8!=0]=0
    d['funding_rate']=funding.ravel()
    return d


def portfolio_checks(df,rules=TradingRules()):
    weights=build_portfolios(df,rules)
    for name,w in weights.items():
        assert w.index.equals(df.index) and np.isfinite(w).all()
        assert w.loc[df['symbol'].isin(['BTCUSDT','ETHUSDT'])].eq(0).all()
        assert w.groupby(df['timestamp']).sum().abs().max()<1e-12
        assert w.abs().groupby(df['timestamp']).sum().max()<=1+1e-12
        assert_series_equal(w,build_portfolios(df,rules,variants=(name,))[name])
    return weights


def weight_causality_checks(df,rules=TradingRules()):
    full=portfolio_checks(df,rules)
    hours=pd.DatetimeIndex(pd.to_datetime(df['timestamp'],utc=True).sort_values().unique())
    cuts=sorted(set([len(hours)//2,int(.85*len(hours)),len(hours)-2]))
    for k in cuts:
        mask=pd.to_datetime(df['timestamp'],utc=True)<=hours[k]
        prefix=build_portfolios(df.loc[mask],rules)
        for name in VARIANTS:
            assert_series_equal(prefix[name],full[name].loc[mask],atol=1e-10,rtol=0)
    changed=df.copy();cut=hours[len(hours)//2]
    future=pd.to_datetime(df['timestamp'],utc=True)>cut
    changed.loc[future,'close']*=1.4
    changed.loc[future,'taker_buy_quote_volume']=changed.loc[future,'quote_volume']
    changed_weights=build_portfolios(changed,rules)
    for name in VARIANTS:
        assert_series_equal(full[name].loc[~future],changed_weights[name].loc[~future],atol=1e-10,rtol=0)
    return 'PASS: index, finite values, leaders zero, neutrality, gross exposure, determinism, prefixes and future perturbation.'


def common_evaluation_sample(df,weights,rules):
    prepared=prepare_signals(df,rules)
    eligible=prepared['ready']
    if not eligible.any():
        raise ValueError('No common post-warm-up history. Need more than 720 hours.')
    start=eligible.index[eligible][0]
    mask=pd.to_datetime(df['timestamp'],utc=True)>=start
    # Preserve full-history feature estimates. Only the PnL reporting sample is
    # trimmed. Every book was zero before this shared eligibility boundary.
    for w in weights.values():
        if (~mask).any():
            assert w.loc[~mask].abs().max()==0
    d=df.loc[mask].copy().reset_index(drop=True)
    trimmed={name:pd.Series(w.loc[mask].to_numpy(),index=d.index,name=name) for name,w in weights.items()}
    tiered_full=pd.Series(tiered_fee_bps(df),index=df.index)
    return d,trimmed,start,tiered_full.loc[mask].to_numpy()


def execution_exposure(df,w,lag_bars):
    held=w.groupby(df['symbol']).shift(lag_bars).fillna(0)
    gross=held.abs().groupby(df['timestamp']).sum()
    changes=held-held.groupby(df['symbol']).shift(1).fillna(0)
    turn=changes.abs().groupby(df['timestamp']).sum()
    return dict(active_hour_fraction=float(gross.gt(1e-12).mean()),
                average_gross=float(gross.mean()),turnover_total=float(turn.sum()),
                turnover_per_day=float(turn.mean()*24))


def economic_grid(df,weights,fees=(0.,3.,5.,10.),tiered_fee_rows=None):
    rows=[];series={};all_metrics={}
    tiered=tiered_fee_bps(df) if tiered_fee_rows is None else tiered_fee_rows
    for name,w in weights.items():
        for lag in (1,2):
            exposure=execution_exposure(df,w,lag)
            for fee_label,fee in [(f'{f:g}bps',f) for f in fees]+[('tiered',tiered)]:
                m=run_backtest(df,w,fee_bps=fee,include_funding=True,lag_bars=lag)
                key=(name,lag,fee_label)
                all_metrics[key]=m;series[key]=m.hourly_returns
                rows.append(dict(portfolio=name,lag_bars=lag,cost=fee_label,
                    sharpe=m.sharpe,sharpe_ci_low=m.sharpe_ci_low,sharpe_ci_high=m.sharpe_ci_high,
                    annualized_mean=m.ann_return,max_drawdown_notional=m.max_drawdown,
                    cumulative_net_pnl=m.net_pnl_cum,gross_price_pnl=m.gross_pnl_cum,
                    fee_cost=m.fee_cost_cum,funding_cost=m.funding_cost_cum,
                    **exposure))
    summary=pd.DataFrame(rows).set_index(['portfolio','lag_bars','cost'])
    hours=pd.DatetimeIndex(sorted(df['timestamp'].unique()))
    returns=pd.DataFrame(series,index=hours)
    returns.columns=pd.MultiIndex.from_tuples(returns.columns,names=['portfolio','lag_bars','cost'])
    return summary,returns,all_metrics


def paired_mean_interval(diff,block=168,reps=1000,seed=20261116):
    """Interval for mean hourly NET PnL difference, not difference in Sharpe."""
    y=np.asarray(diff,dtype=float)
    if not np.isfinite(y).all():
        raise ValueError('Paired PnL comparisons require identical finite hourly samples')
    n=len(y)
    if n<2*block:
        return np.nan,np.nan
    rng=np.random.default_rng(seed);full,rem=divmod(n,block)
    a=np.r_[y,y[:block]];cs=np.r_[0.,np.cumsum(a)]
    starts=rng.integers(0,n,size=(reps,full))
    sums=(cs[starts+block]-cs[starts]).sum(axis=1)
    if rem:
        s=rng.integers(0,n,size=reps);sums+=cs[s+rem]-cs[s]
    return tuple(np.quantile(sums/n,[.025,.975]))


def economic_pairs(returns):
    rows=[]
    comparisons=[('catchup','reversal'),('catchup_strong','catchup'),('catchup_strong','reversal')]
    for lag in (1,2):
        for j,(a,b) in enumerate(comparisons):
            diff=returns[(a,lag,'5bps')]-returns[(b,lag,'5bps')]
            lo,hi=paired_mean_interval(diff,seed=20261116+100*lag+j)
            rows.append(dict(lag_bars=lag,comparison=a+' minus '+b,
                mean_hourly_net_difference=diff.mean(),annualized_mean_difference=diff.mean()*8760,
                annualized_ci_low=lo*8760,annualized_ci_high=hi*8760))
    return pd.DataFrame(rows)


def breakeven_fees(returns):
    rows=[]
    for name in VARIANTS:
        for lag in (1,2):
            r0=returns[(name,lag,'0bps')]
            r5=returns[(name,lag,'5bps')]
            total_turn=(r0-r5).sum()/.0005
            # At constant weights and linear fees, break-even SUM/MEAN PnL:
            # gross price PnL - funding - fee*turnover = 0.
            raw_bps=r0.sum()/total_turn*10000 if total_turn>1e-12 else np.nan
            rows.append(dict(portfolio=name,lag_bars=lag,
                breakeven_fee_bps=raw_bps if raw_bps>=0 else np.nan,
                note=('No nonnegative fee breaks even' if raw_bps<0 else
                      'No turnover / undefined' if not np.isfinite(raw_bps) else
                      'Linear fee-only estimate; funding included; no spread/impact')))
    return pd.DataFrame(rows)


def file_hash(path):
    digest=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()


## 2. Settings and the exact planned experiment
`RUN_BACKTEST=False` is the default so Run All first prints the plan and runs only
synthetic checks. Copy the plan into your PR, then set the flag True and restart/run
all for the real economic experiment. This is a workflow step, not fresh untouched
preregistration: disclose your existing IC analyses and research-selected 8h choice.


In [ ]:
USE_REAL_DATA = True
RUN_BACKTEST = False
RULES=TradingRules()
DATA_PATH=ROOT/'data'/'train_1h.parquet'
print('Rules:',asdict(RULES))
print('Economic performance run enabled:',RUN_BACKTEST)

PLAN="""Track 03 economic experiment after original/corrected IC analysis and beta diagnostics.
Three fixed books: reversal, ungated beta-adjusted catchup, strong-flow catchup.
Original signal settings: gap4h, beta168h, impulse span4, prior720h percentile .9.
Full histories required. Shared eligibility, 18 alts only; BTC/ETH always zero.
Weights: per-hour alt-only cross-sectional z-score, clip3, neutralize and normalize.
Decisions every 8 epoch-hours (UTC open labels 00,08,16); known after that bar closes.
8h is selected by the recorded lag-1 strong-hour IC-peak rule, not proven optimal.
Carry issued weights between decisions. Existing positions may persist over cycles.
Per-symbol deadband .015 against previous issued weights; then re-neutralize and
normalize active gross to1. Projection may move other names; count resulting turnover.
At scheduled decisions, gated catchup must flatten if flow is weak or unknown.
No immediate exit between decisions. Flattening bypasses the band. Unknown features
also flatten at scheduled decisions. No terminal liquidation based on dataset length.
Same post-warm-up evaluation dates for all books. No leverage/risk matching of the
gated book; show actual average gross and active fraction alongside Sharpe/PnL.
Official run_backtest: lag_bars1 is close(t) entry, next close-to-close return;
lag_bars2 adds one real hour. Run all three at both delays, all fees 0/3/5/10bps and
the repository's tiered schedule. Funding included in every row, including zero fees.
Tiered fees retain the full panel's historical-volume estimates before sample trim.
Report Sharpe and its repository CI, net/gross-price PnL, fees, funding, notional
drawdown, turnover and exposure. Primary economic view: 5bps, lag_bars2. Report all
rows; do not select a more flattering cost/delay. Show paired mean net-PnL differences
with 168h circular-block bootstrap intervals (1000 draws), not Sharpe differences.
Pointwise intervals do not correct for multiple comparisons or prior research.
Do not use sprint holdout/forward data. Keep REJECT/negative results. No automatic
strategy substitution or repeated tuning. Main declared strategy is catchup_strong.
"""
print(PLAN)


## 3. Meaningful correctness tests
These check final portfolio constraints and causality, the fixed clock and gate,
deadband semantics, original index alignment and no input mutation. A tiny independent
ledger verifies the repository's price-return, delay, fee and asymmetric funding
timing. Higher fees are checked against identical books. Prefix tests ensure no
future-dependent terminal liquidation. No real market performance is tested here.


In [ ]:
import unittest
from src.features import rebalance_deadband


class EconomicTests(unittest.TestCase):
    def setUp(self):
        self.rules=TradingRules(beta_window=24,quantile_window=48)
        self.df=synthetic_economic_panel(180)

    def test_constraints_and_entry_point(self):
        books=portfolio_checks(self.df,self.rules)
        self.assertTrue(any(w.abs().max()>0 for w in books.values()))
        assert_series_equal(compute_weights(self.df),
            build_portfolios(self.df,TradingRules(),variants=('catchup_strong',))['catchup_strong'])

    def test_index_shuffle_and_no_mutation(self):
        before=self.df.copy(deep=True)
        shuffled=self.df.sample(frac=1,random_state=8)
        a=build_portfolios(self.df,self.rules);b=build_portfolios(shuffled,self.rules)
        for name in VARIANTS:assert_series_equal(a[name].loc[shuffled.index],b[name])
        pd.testing.assert_frame_equal(self.df,before)

    def test_final_weight_causality(self):
        self.assertTrue(weight_causality_checks(self.df,self.rules).startswith('PASS'))

    def test_hold_schedule_and_gated_exit(self):
        index=pd.date_range('2026-01-01',periods=25,freq='h',tz='UTC')
        target=np.tile(neutral_unit_gross(np.arange(18)),(25,1))
        strong=np.ones(25,dtype=bool);strong[1:9]=False
        book=scheduled_book(target,np.ones(25,dtype=bool),strong,index,replace(self.rules,deadband=0),gate=True)
        np.testing.assert_allclose(book[:8],target[:8])
        np.testing.assert_allclose(book[8:16],0)
        np.testing.assert_allclose(book[16:24],target[16:24])

    def test_no_terminal_liquidation_lookahead(self):
        full=build_portfolios(self.df,self.rules)
        cutoff=sorted(self.df['timestamp'].unique())[-3]
        mask=self.df['timestamp']<=cutoff
        short=build_portfolios(self.df.loc[mask],self.rules)
        for name in VARIANTS:assert_series_equal(full[name].loc[mask],short[name])

    def test_deadband_matches_shared_single_step(self):
        prev=np.array([.10,-.10,.03,-.03]);target=np.array([.11,-.13,.03,-.01])
        d=pd.DataFrame({'symbol':['a','b','c','d'],'timestamp':pd.Timestamp('2026-01-01',tz='UTC')})
        delta=pd.Series(target-prev,index=d.index)
        expected=prev+rebalance_deadband(d,delta,buffer=.015).to_numpy()
        np.testing.assert_allclose(threshold_step(prev,target,.015),expected)

    def test_empty_and_invalid_inputs(self):
        for w in build_portfolios(self.df.iloc[:0],self.rules).values():self.assertTrue(w.empty)
        with self.assertRaises(ValueError):build_portfolios(self.df.iloc[1:],self.rules)

    def test_backtest_timing_fees_funding_against_manual_ledger(self):
        times=pd.date_range('2026-01-01',periods=10,freq='h',tz='UTC')
        d=pd.MultiIndex.from_product([times,['a','b']],names=['timestamp','symbol']).to_frame(index=False)
        prices=np.column_stack([100*1.01**np.arange(10),100*.995**np.arange(10)])
        d['close']=prices.ravel();d['funding_rate']=0.
        d.loc[d['timestamp'].isin([times[6],times[7]]) & d['symbol'].eq('a'),'funding_rate']=.001
        matrix=np.zeros((10,2));matrix[1:5]=[.5,-.5];matrix[5:]=[-.5,.5]
        w=pd.Series(matrix.ravel(),index=d.index)
        for lag in (1,2):
            m=run_backtest(d,w,fee_bps=5,lag_bars=lag)
            ledger=[]
            for t in range(10):
                held=matrix[t-lag] if t>=lag else np.zeros(2)
                prior=matrix[t-lag-1] if t>=lag+1 else np.zeros(2)
                ret=prices[t]/prices[t-1]-1 if t else np.zeros(2)
                funding=np.array([.001,0.]) if t in (6,7) else np.zeros(2)
                ledger.append(np.sum(held*ret)-np.abs(held-prior).sum()*.0005-np.sum(prior*funding))
            np.testing.assert_allclose(m.hourly_returns,ledger,atol=1e-14)

    def test_higher_fees_reduce_same_book_pnl(self):
        books=build_portfolios(self.df,self.rules)
        w=books['catchup']
        a=run_backtest(self.df,w,fee_bps=0)
        b=run_backtest(self.df,w,fee_bps=5)
        c=run_backtest(self.df,w,fee_bps=10)
        np.testing.assert_allclose(a.hourly_returns-c.hourly_returns,
            2*(a.hourly_returns-b.hourly_returns),atol=1e-14)
        self.assertGreater((a.hourly_returns-b.hourly_returns).sum(),0)

    def test_common_evaluation_preserves_post_warmup_ledger(self):
        books=build_portfolios(self.df,self.rules)
        d,weights,start,fees=common_evaluation_sample(self.df,books,self.rules)
        self.assertEqual(len(fees),len(d))
        mask=np.array(sorted(self.df['timestamp'].unique()))>=start
        for name in VARIANTS:
            full=run_backtest(self.df,books[name],fee_bps=5)
            trimmed=run_backtest(d,weights[name],fee_bps=5)
            np.testing.assert_allclose(full.hourly_returns[mask],trimmed.hourly_returns,atol=1e-14)

    def test_paired_interval_identical_books_is_zero(self):
        low,high=paired_mean_interval(np.zeros(500),block=24,reps=100)
        self.assertEqual(low,0);self.assertEqual(high,0)


economic_tests=unittest.TextTestRunner(verbosity=1).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(EconomicTests))
assert economic_tests.wasSuccessful(),'Economic implementation checks failed; stop before interpreting results.'


## 4. Load training data, build weights and check them
The first pass is skipped until RUN_BACKTEST=True. No training file is read while
the flag is False. Feature history is calculated on the full panel. Reporting then
starts at the shared eligibility boundary (normally hour720). Every book is zero
before that boundary. Initial beta/flow history is never reconstructed from the
trimmed sample. Tiered fees also retain their full historical information.


In [ ]:
summary=None;returns=None;weights=None;pair_table=None;breakeven=None
audit_window_reference=None;full_df=None;status=None;eval_start=None
label='REAL training economic experiment' if USE_REAL_DATA else 'SYNTHETIC: not a market result'
if RUN_BACKTEST:
    full_df=pd.read_parquet(DATA_PATH) if USE_REAL_DATA else synthetic_economic_panel()
    full_df=full_df.sort_values(['timestamp','symbol']).reset_index(drop=True)
    if 'funding_rate' not in full_df:
        raise ValueError('The economic experiment requires actual funding_rate; do not invent zeros for real data.')
    if not np.isfinite(full_df['funding_rate'].to_numpy(dtype=float)).all():
        raise ValueError('Non-finite funding observations')
    status=weight_causality_checks(full_df,RULES)
    weights=build_portfolios(full_df,RULES)
    eval_df,eval_weights,eval_start,eval_tiered=common_evaluation_sample(full_df,weights,RULES)
    print(label,full_df.shape)
    print(status)
    print('Shared reporting sample:',eval_start,'to',eval_df['timestamp'].max())
    print('Hourly observations:',eval_df['timestamp'].nunique())
else:
    print('Performance skipped. Record PLAN in your PR, set RUN_BACKTEST=True, then restart/run all.')


## 5. Costs, delay and exposure — the actual trading experiment
There are 30 fixed backtests: 3 books × 2 execution delays × 5 cost rows.

- The **primary economic view is 5bps with lag_bars=2**, adding an hour of real delay.
- Funding is included at fee=0 too. Zero-fee PnL therefore differs from gross price PnL.
- Costs apply to the absolute change of every held weight (both sides of turnover).
- Active books have long weight .5 and short weight -.5, so gross1 and net0.
- Gating changes time in the market as well as turnover. A different Sharpe does
  not isolate a pure prediction improvement; compare net PnL and exposure as well.
- Cumulative PnL and drawdown use the harness's additive fixed-notional accounting,
  NOT compounded account growth. No spread, impact, margin liquidation or queue model.
- The engine charges no artificial final close fee; positions are not forcibly
  liquidated at a future-known dataset endpoint. Retain this limitation in reporting.

The repository Sharpe intervals use its existing 24-hour block routine. The paired
mean-PnL intervals later use 168-hour blocks. These assess different quantities and
remain approximate pointwise intervals; neither certifies an edge.


In [ ]:
if RUN_BACKTEST:
    summary,returns,metrics=economic_grid(eval_df,eval_weights,tiered_fee_rows=eval_tiered)
    with pd.option_context('display.max_rows',40):
        display(summary[['sharpe','sharpe_ci_low','sharpe_ci_high','cumulative_net_pnl',
            'max_drawdown_notional','turnover_per_day','average_gross','active_hour_fraction']].round(5))
    print('PRIMARY ECONOMIC VIEW: 5bps, extra-hour delayed execution')
    display(summary.xs((2,'5bps'),level=('lag_bars','cost'))[
        ['sharpe','cumulative_net_pnl','gross_price_pnl','fee_cost','funding_cost',
         'turnover_per_day','average_gross','active_hour_fraction']].round(5))


## 6. Return path and sensitivity to fees
These plots show what the portfolios earned under the model. They do not turn an
in-sample result into validation. Inspect whether gains depend on a short period.
The gated line is not levered up to compensate for spending more time flat.


In [ ]:
if RUN_BACKTEST:
    from matplotlib import dates as mdates
    fig,axes=plt.subplots(1,2,figsize=(13,3.8),sharey=True)
    for ax,lag in zip(axes,(1,2)):
        for name in VARIANTS:
            ax.plot(returns.index,returns[(name,lag,'5bps')].cumsum(),label=name,lw=1)
        ax.axhline(0,color='black',lw=.5);ax.set_title(f'5bps, lag_bars={lag}')
        locator=mdates.AutoDateLocator(minticks=3,maxticks=5)
        ax.xaxis.set_major_locator(locator);ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        ax.legend(fontsize=8)
    axes[0].set_ylabel('Cumulative net PnL / initial notional (additive)')
    fig.suptitle(label);fig.tight_layout();plt.show()
    fig,axes=plt.subplots(1,2,figsize=(12,3.5))
    for ax,lag in zip(axes,(1,2)):
        for name in VARIANTS:
            y=[summary.loc[(name,lag,f'{fee}bps'),'cumulative_net_pnl'] for fee in (0,3,5,10)]
            ax.plot([0,3,5,10],y,'o-',label=name)
        ax.axhline(0,color='black',lw=.5);ax.set_xlabel('Flat fee (bps per unit turnover)')
        ax.set_title(f'Fee decay, lag_bars={lag}');ax.legend(fontsize=8)
    axes[0].set_ylabel('Cumulative net PnL / initial notional')
    fig.tight_layout();plt.show()


## 7. Paired economic comparisons, not just separate Sharpe ratios
Subtract hourly NET PnL on identical dates at 5bps. The interval describes the mean
of this difference, then is multiplied by8760 for readability. It is **not a
difference in Sharpe ratios** and not a compounded annual-growth estimate.

- catchup minus reversal: economic contribution of beta adjustment.
- catchup_strong minus catchup: effect of the flow gate plus changed exposure/turnover.
- catchup_strong minus reversal: complete proposed design versus the simple benchmark.

Below-zero intervals favor the second book; above-zero favor the first; intervals
including zero are inconclusive. Do not hunt among six comparisons for one positive.


In [ ]:
if RUN_BACKTEST:
    pair_table=economic_pairs(returns)
    display(pair_table.round(6))
    breakeven=breakeven_fees(returns)
    display(breakeven.round(4))


## 8. Why the official audit may report slightly different summaries
This notebook removes the shared initial zero-position warm-up from its main
performance sample. The audit summarizes the full input panel, including warm-up.
Trades and post-warm-up hourly PnL are identical, but annualization/Sharpe averages
can differ. Here is the full-panel 5bps/default-delay reference for all three books.
Do not compare different reporting periods without saying so.


In [ ]:
if RUN_BACKTEST:
    rows=[]
    for name,w in weights.items():
        m=run_backtest(full_df,w,fee_bps=5,lag_bars=1)
        rows.append(dict(portfolio=name,full_panel_sharpe=m.sharpe,
                         full_panel_net_pnl=m.net_pnl_cum,full_panel_turnover=m.mean_turnover))
    audit_window_reference=pd.DataFrame(rows).set_index('portfolio')
    display(audit_window_reference)


## 9. Save the full record, including negative results
Runs get fresh directories. Metadata hashes the data, strategy and relevant harness
files. CSVs retain every cost/delay/book, not only a preferred result. A plan-only run
saves no performance numbers and does not read real data. The code never posts a PR.


In [ ]:
from datetime import datetime,timezone
from uuid import uuid4
OUT=ROOT/'results'/('track03_economics_'+('real_' if USE_REAL_DATA else 'synthetic_')+
    ('run_' if RUN_BACKTEST else 'plan_')+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid4().hex[:6])
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'trading_plan.txt').write_text(PLAN,encoding='utf-8')
source_files=['strategies/janice_leadlag.py','src/contrib/janice.py','src/features.py','src/backtest.py','src/stats.py']
metadata=dict(created_utc=datetime.now(timezone.utc).isoformat(),rules=asdict(RULES),
    performance_executed=RUN_BACKTEST,data_mode=label,evaluation_start=str(eval_start),
    data_sha256=file_hash(DATA_PATH) if RUN_BACKTEST and USE_REAL_DATA else None,
    synthetic_seed=41 if RUN_BACKTEST and not USE_REAL_DATA else None,
    source_sha256={path:file_hash(ROOT/path) for path in source_files},
    python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,
    final_weight_checks=status,declared_strategy='catchup_strong',
    warning='Exploratory training-panel economic experiment; not an audit verdict or sealed evaluation.')
if RUN_BACKTEST:
    summary.to_csv(OUT/'all_portfolio_cost_delay_results.csv')
    returns.to_csv(OUT/'hourly_net_pnl.csv')
    pair_table.to_csv(OUT/'paired_net_pnl_comparisons.csv',index=False)
    breakeven.to_csv(OUT/'breakeven_fees.csv',index=False)
    audit_window_reference.to_csv(OUT/'full_panel_reference.csv')
    pd.DataFrame(weights).to_csv(OUT/'full_panel_issued_weights.csv',index_label='input_index')
(OUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('Saved:',OUT)


## 10. Interpret and finish the sprint submission
| Finding | Interpretation / action |
|---|---|
| Positive zero-fee PnL, negative at5bps | Prediction does not cover modeled fees at these rules; report it |
| Default execution works, extra-hour delay fails | Timing-sensitive result; do not treat close fills as robust execution evidence |
| Beta catches up on IC but loses on PnL | Magnitudes, position sizing, turnover or funding changed the economic result |
| Gate improves Sharpe but reduces PnL/time active | Inspect exposure and risk; this is not proof that flow improves prediction |
| All books lose after costs | Valid negative research result; do not force a profitable version by repeated tuning |
| One book profits | Training-panel evidence only; inspect uncertainty, drawdown and model limits |

Do not automatically replace the assigned strategy with whichever benchmark wins.
The supplied audit entry point remains `catchup_strong`. If you want a different
submitted specification, describe the change and its research history in the PR
and agree on scope where needed. Reversal is a benchmark, not a hidden replacement.

When ready, run FROM THE REPOSITORY ROOT (outside the notebook):
```
python -m pytest -q
python -m src.audit strategies/janice_leadlag.py --seed 20261116 --n-random-cuts 60
```
The audit also tests the purged split, lag, null and wick criteria not decided by
this notebook. REJECT exits1 and may be an honest performance result. Read each
check: leakage, crashes or invalid weights require correction. Include the audit
output, the trading plan and actual test results in the PR, plus the required
100–200-word interpretation. Keep the earlier IC tables and diagnostics as evidence.

**Deadband implementation note:** to preserve neutrality after each decision, the
strategy implements the same per-symbol threshold rule as the shared
`rebalance_deadband`, then recenters/rescales and remembers the final issued weights.
Using the unprojected internal state after normalization would not represent the
actual issued portfolio. The equivalent threshold rule is tested against the shared
helper. Neutralization can create smaller trades in other coins; actual turnover
from those trades is charged by the backtest rather than assumed away.
